# ML model for Keyword Classification - Non-tech Notebook
This notebook provides a guide for non-technical audiences on how to use a trained machine learning model.

The prediction workflows are embedded within the `KeywordClassifierPipeline` class, where three key parameters: `usePretrainedModel`, `model_name`, and `isDataChanged`, control the tasks in the pipeline:

1. **`usePretrainedModel`**  
   - Set `usePretrainedModel` to `True` to use an existing pretrained model specified by `model_name`, which will skip the training tasks. The pipeline will make predictions using the pretrained model.
   - **Note:** If `usePretrainedModel` is set to `True`, then `model_name` **must not** be `None`.
   - If `usePretrainedModel` is set to `False`, the pipeline will check the value of `isDataChanged` to determine the next steps.

2. **`model_name`**  
   - This parameter specifies the name of the pretrained model file (excluding the file extension) saved within the project. 
   - If a new model needs to be trained, `model_name` can be `None` so a default name will assigned.

3. **`isDataChanged`**  
   - This parameter indicates if there have been significant changes to the raw data, such as new data being added.
   - Set `isDataChanged` to `True` if the data has changed, prompting the pipeline to:
     - Re-query the data from Elasticsearch.
     - Select a new sample set from the updated data.
     - Use this new sample set for training a new model for future predictions.

There are four typical scenarios to consider:

1. **Scenario 1: Using a Pretrained Model**: A pretrained model is available, and no retraining is necessary. We can directly use the model for prediction tasks. `usePretrainedModel=True, isDataChanged=False, model_name='saved_model_name'`.
2. **Scenario 2: Hyperparameter Tuning**: A pretrained model is available, but we want to improve performance by adjusting hyperparameters. This requires retraining the model with the new settings. `usePretrainedModel=False, isDataChanged=False, model_name='saved_model_name'`.
3. **Scenario 3: Training a New Model**: No pretrained model is available, so we need to prepare a sample set, train a new model, and evaluate its performance from the beginning. `usePretrainedModel=False, isDataChanged=False, model_name=None`.
4. **Scenario 4: Update Sample Set Due to Data Changes**: A pretrained model is available, but the data has changed significantly. In this case, we need to update the sample set (not recommend, if the data changed significantly, we want to retrain the model based on new data). `usePretrainedModel=True, isDataChanged=True, model_name='saved_model_name'`.



In [1]:
# add module folder path for use in notebook
import sys
sys.path.append('../data_discovery_ai')

In [2]:
import model.keywordModel as model
import utils.preprocessor as preprocessor

c:\Users\yhu12\AppData\Local\miniforge3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# add pipeline folder to system path for use in notebook
sys.path.append('../')

In [4]:
# define a abstract for example
abstract = """
                    Ecological and taxonomic surveys of hermatypic scleractinian corals were carried out at approximately 100 sites around Lord Howe Island. 
                    Sixty-six of these sites were located on reefs in the lagoon, which extends for two-thirds of the length of the island on the western side. 
                    Each survey site consisted of a section of reef surface, which appeared to be topographically and faunistically homogeneous. 
                    The dimensions of the sites surveyed were generally of the order of 20m by 20m. 
                    Where possible, sites were arranged contiguously along a band up the reef slope and across the flat. 
                    The cover of each species was graded on a five-point scale of percentage relative cover. 
                    Other site attributes recorded were depth (minimum and maximum corrected to datum), slope (estimated), substrate type, total estimated cover of soft coral and algae (macroscopic and encrusting coralline). 
                    Coral data from the lagoon and its reef (66 sites) were used to define a small number of site groups which characterize most of this area.
                    Throughout the survey, corals of taxonomic interest or difficulty were collected, and an extensive photographic record was made to augment survey data. 
                    A collection of the full range of form of all coral species was made during the survey and an identified reference series was deposited in the Australian Museum.
                    In addition, less detailed descriptive data pertaining to coral communities and topography were recorded on 12 reconnaissance transects, the authors recording changes seen while being towed behind a boat.
                    The purpose of this study was to describe the corals of Lord Howe Island (the southernmost Indo-Pacific reef) at species and community level using methods that would allow differentiation of community types and allow comparisons with coral communities in other geographic locations.
                    """
                    

## Scenario 1: Using a Pretrained Model
In this case, we don't need to update the sample data nor re-train the model. Thus, we can directly use the `make_prediction` method from `KeywordClassifierPipeline` class. When init the `KeywordClassifierPipeline` object, make sure set `usePretrainedModel=True`, and `model_name` is correct.

In [5]:
# identify the pretrained model to be used
pretrained_model_name = "pretrainedKeyword4demo"

In [6]:
from pipeline import KeywordClassifierPipeline

scenario1_pipeline = KeywordClassifierPipeline(usePretrainedModel=True, model_name=pretrained_model_name, isDataChanged=False)
predictions = scenario1_pipeline.make_prediction(description=abstract)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step
AODN Discovery Parameter Vocabulary:Biotic taxonomic identification


Using `pipeline.pipeline` function has the same effect:

In [7]:
from pipeline import pipeline
scenario1_pipeline = pipeline(isDataChanged=False, usePretrainedModel=True, selected_model=pretrained_model_name, description=abstract)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
AODN Discovery Parameter Vocabulary:Biotic taxonomic identification


## Scenario 2: Hyperparameter Tuning
In cases where the model requires retraining, we can use adjusted hyperparameters to optimize performance. These hyperparameters are predefined in the file `data_discovery_ai/common/keyword_classification_parameters.ini` and are applied across data preprocessing, training, and prediction modules.

**Note:** Changes to these parameters can affect both model performance and prediction accuracy.

In [8]:
scenario2_pipeline = KeywordClassifierPipeline(isDataChanged=False, usePretrainedModel=False, model_name="retrain_model")

We first load sample set which is used for training the model.

In [9]:
sampleSet = preprocessor.load_from_file("../data_discovery_ai/input/keyword_sample.pkl")
sampleSet

,id,title,description,keywords,embedding
0,52b58d9a-a0b4-4396-be8e-a9e5e2b493f0,IMOS SOOP Underway Data from AIMS Vessel RV So...,'Ships of Opportunity' (SOOP) is a facility of...,[AODN Discovery Parameter Vocabulary:Turbidity...,"[-0.57627493, -0.52038336, 0.14258689, -0.0053..."
1,52c92036-cea9-4b1a-b4f0-cc94b8b5df98,IMOS - SRS - SST - L3C - NOAA 19 - 3 day - day...,"This is a single sensor, multiple swath SSTski...","[AODN Instrument Vocabulary:radiometers, AODN ...","[-0.7428907, -0.006864315, 0.20524804, -0.6074..."
2,52e8e882-5108-4295-b336-e97c11af7ad4,Sea Water Temperature Logger Data at Taure Ree...,This data set was collected by one or more tem...,[AODN Discovery Parameter Vocabulary:Depth of ...,"[-0.36630422, -0.2719815, 0.26079246, -0.06611..."
3,52f09a23-63a2-4c14-8b3b-1fc7c8167281,IMOS - ACORN - Turquoise Coast HF ocean radar ...,The Turquoise Coast (TURQ) HF ocean radar syst...,[AODN Discovery Parameter Vocabulary:Current s...,"[-1.013301, -0.28140602, 0.2872767, -0.2471972..."
4,532138db-7c8f-4346-82cf-04d16e4d662d,WA Marine Futures Project - reef habitat,The Marine Futures Project was designed to ben...,"[AODN Platform Vocabulary:research vessel, AOD...","[-0.75856686, -0.5568418, -0.01814289, -0.0615..."
...,...,...,...,...,...
1626,fefe36ac-f4fc-5eae-e044-00144fdd4fa6,Petrel Sub-basin Marine Survey (GA-0335 / SOL5...,The Petrel Sub-basin Marine Environmental Surv...,"[AODN Platform Vocabulary:research vessel, AOD...","[-1.1288652, -0.2893246, 0.027036984, -0.24384..."
1627,fed52ea8-bde9-4126-aa3d-69431fce5694,Port Curtis Integrated Monitoring Program - PC...,This data set was collected by sensors deploye...,[AODN Discovery Parameter Vocabulary:Dissolved...,"[-0.5563927, -0.6243812, 0.3688077, -0.1797219..."
1628,fcd7a039-2134-4761-ad08-ec42b8e05610,IMOS SOOP Underway Data from AIMS Vessel RV So...,'Ships of Opportunity' (SOOP) is a facility of...,[AODN Discovery Parameter Vocabulary:Turbidity...,"[-0.57493657, -0.5267085, 0.12662068, -0.01271..."
1629,fbe4dbce-3435-48df-a054-f0e399886e2b,IMOS - SRS - SST - L3S - Single Sensor - 14 da...,This is a multi-sensor SSTskin product for fou...,"[AODN Instrument Vocabulary:radiometers, AODN ...","[-0.8789551, -0.2196846, 0.28057387, -0.572204..."


After adjusting hyperparameter values, let's say change learning rate from `0.001` to `0.01`.

In [10]:
train_test_data = scenario2_pipeline.prepare_train_test_sets(sampleSet)
scenario2_pipeline.train_evaluate_model(train_test_data)
scenario2_pipeline.make_prediction(abstract)

 ======== After Resampling ========
Total samples: 3100
Dimension: 768
No. of labels: 498
X resampled set size: 3100
Y resampled set size: 3100
 ======== After Resampling ========
Total samples: 106144
Dimension: 768
No. of labels: 498
X resampled set size: 106144
Y resampled set size: 106144


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 498)            │        64,242 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 162,674 (635.45 KB)

 Trainable params: 162,674 (635.45 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
2654/2654 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.1344 - loss: 3.6761e-04 - precision: 0.4815 - recall: 0.5425 - val_accuracy: 0.0000e+00 - val_loss: 0.0092 - val_precision: 0.6500 - val_recall: 0.2307 - learning_rate: 0.0010
Epoch 2/100
2654/2654 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.2632 - loss: 4.4549e-05 - precision: 0.9047 - recall: 0.9360 - val_accuracy: 0.0203 - val_loss: 0.0082 - val_precision: 0.5222 - val_recall: 0.5099 - learning_rate: 0.0010
Epoch 3/100
2654/2654 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.2742 - loss: 3.1830e-05 - precision: 0.9270 - recall: 0.9493 - val_accuracy: 0.0729 - val_loss: 0.0080 - val_precision: 0.5149 - val_recall: 0.5252 - learning_rate: 0.0010
Epoch 4/100
2654/2654 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.2769 - loss: 2.7556e-05 - precision: 0.9338 - recall: 0.9527 - val_accuracy: 0.0553 - val_loss: 0.0073 - val_precision: 0.4905 - val_recall: 0.7200 - learning_rate: 0.0010
Epoch 5/100
2654/2654 ━━━━━━

'AODN Discovery Parameter Vocabulary:Biotic taxonomic identification'

## Scenario 3: Training a New Model
In this case, we want to train a new model no matter because there is no pretrained model before, or data chenged.

In [ ]:
scenario3_pipeline = KeywordClassifierPipeline(isDataChanged=True, usePretrainedModel=False, model_name="retrain_pipeline3")
raw_data = scenario3_pipeline.fetch_raw_data()
sampleSet = scenario3_pipeline.prepare_sampleSet(raw_data=raw_data)
train_test_data = scenario3_pipeline.prepare_train_test_sets(sampleSet)
scenario3_pipeline.train_evaluate_model(train_test_data)

scenario3_pipeline.make_prediction(abstract)

## Prepare Dataset
This section introduces how to load the target and sample sets, snippets to explore these sets, and prepare and test sets.